# Tutorial 1: Your First Quantum Chemistry Calculation

**Welcome to your first quantum chemistry calculation!**

## 🎓 Learning Objectives

By the end of this tutorial, you will:
- ✅ Submit a calculation to the QuantUI cluster  
- ✅ Monitor job progress  
- ✅ Interpret basic results (SCF energy, convergence)  
- ✅ Understand what a Hartree-Fock calculation does  
- ✅ Visualize a molecule in 3D

## ⏱️ Estimated Time
**15-20 minutes** (including job queue time)

## 📚 Prerequisites
- Access to the cluster
- QuantUI environment activated
- Basic chemistry knowledge (atoms, molecules)

---

**Let's get started!** 🚀

---

## Step 1: Setup

First, we need to initialize the QuantUI system.

### What's happening here?
- Loading QuantUI modules
- Connecting to in-session runner
- Setting up your user workspace

### Run the cell below ⬇️

In [ ]:
# Initialize QuantUI
import os
import sys
from pathlib import Path

# Import QuantUI modules
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import quantui
from quantui import config

# Detect username
username = quantui.get_username()
print(f"✓ Welcome, {username}!")

# Initialize job manager
print(f"✓ Job manager initialized")
print(f"✓ Your calculations will be saved to: {job_manager.calculations_dir}")

# Global state
current_molecule = None

print("\n" + "="*60)
print("✓ Setup complete! Ready for your first calculation.")
print("="*60)
# QuantUI — in-session calculation imports
from quantui import (
    Molecule, parse_xyz_input,
    MOLECULE_LIBRARY, SUPPORTED_METHODS, SUPPORTED_BASIS_SETS,
    session_can_handle, get_session_resources,
    PUBCHEM_AVAILABLE, VISUALIZATION_AVAILABLE, ASE_AVAILABLE,
)
try:
    from quantui import run_in_session, SessionResult
    PYSCF_AVAILABLE = True
except (ImportError, AttributeError):
    PYSCF_AVAILABLE = False
    print("PySCF not available — calculations will not run.")
    print("PySCF requires Linux, macOS, or WSL.")


---

## Step 2: Define Your Molecule - Water (H₂O)

We'll start with water - one of the most important molecules in chemistry!

### About Water (H₂O)
- **Chemical formula:** H₂O
- **Atoms:** 1 oxygen + 2 hydrogen
- **Shape:** Bent (not linear!)
- **Bond angle:** ~104.5°
- **Why it matters:** Essential for life, unique properties

### XYZ Coordinates Explained
Each line has: `ATOM  X  Y  Z`
- **ATOM**: Element symbol (O, H, C, etc.)
- **X, Y, Z**: Position in 3D space (in Ångstroms)
- (0,0,0) is arbitrary origin point

### Run the cell below to define water ⬇️

In [ ]:
# Define water molecule coordinates
water_xyz = """O  0.000  0.000  0.000
H  0.757  0.587  0.000
H -0.757  0.587  0.000"""

# Molecular properties
charge = 0          # Neutral molecule (no net charge)
multiplicity = 1    # Singlet state (all electrons paired)

print("Water Molecule (H₂O)")
print("="*40)
print(water_xyz)
print("="*40)
print(f"Charge: {charge}")
print(f"Multiplicity: {multiplicity}")
print("\n💡 Tip: Multiplicity = 1 means all electrons are paired")

### ✅ Checkpoint 1

**Before proceeding, make sure:**
- [ ] You see 3 lines of coordinates (O, H, H)
- [ ] Charge is 0 (neutral)
- [ ] Multiplicity is 1 (paired electrons)

**Questions to think about:**
1. Why do we need 3D coordinates for a calculation?
2. What would happen if we used the wrong charge?

---

## Step 3: Create the Molecule Object

Now we'll create a QuantUI `Molecule` object that validates our input.

### What's being validated?
- Atom symbols are valid elements
- Coordinates are numbers
- Charge and multiplicity are compatible
- Geometry is reasonable (atoms aren't too close)

### Run the cell below ⬇️

In [ ]:
# Parse coordinates and create molecule
atoms, coordinates = parse_xyz_input(water_xyz)

# Create molecule object
current_molecule = Molecule(
    atoms=atoms,
    coordinates=coordinates,
    charge=charge,
    multiplicity=multiplicity
)

# Display molecule info
print("✓ Molecule created successfully!")
print("\n📊 Molecule Summary:")
print(f"   Formula: {current_molecule.get_formula()}")
print(f"   Total atoms: {len(atoms)}")
print(f"   Total electrons: {current_molecule.get_num_electrons()}")
print(f"   Charge: {charge}")
print(f"   Multiplicity: {multiplicity}")

# Try to visualize (if available)
try:
    from quantui import display_molecule, PY3DMOL_AVAILABLE
    if PY3DMOL_AVAILABLE:
        print("\n🔬 3D Visualization:")
        display_molecule(current_molecule, mode="ball+stick", show_info=False)
    else:
        print("\n💡 Install py3Dmol to see 3D visualization")
except:
    print("\n💡 Visualization not available in this environment")

### ✅ Checkpoint 2

**Verify your molecule:**
- Formula should be: **H2O**
- Total atoms: **3**
- Total electrons: **10** (8 from O, 1 each from H)
- If you see 3D visualization, molecule should look bent, not linear

**Understanding electrons:**
- Oxygen: 8 electrons (atomic number = 8)
- Hydrogen: 1 electron each (atomic number = 1)
- Total: 8 + 1 + 1 = 10 electrons

---

## Step 4: Configure the Calculation

Now we'll set up the quantum chemistry calculation.

### Method: RHF (Restricted Hartree-Fock)
- **RHF** = Restricted Hartree-Fock
- Good for closed-shell molecules (all electrons paired)
- Treats electron-electron interactions approximately
- Fast and reliable for small molecules

### Basis Set: 6-31G
- **Basis set** = Mathematical functions to describe electrons
- **6-31G** = Medium-quality basis (good balance of speed/accuracy)
- Larger basis = more accurate but slower
- Common choices: STO-3G (small), 6-31G (medium), cc-pVDZ (large)

### Run the cell below ⬇️

In [ ]:
# Calculation settings
method = "RHF"      # Hartree-Fock method
basis = "6-31G"     # Basis set
job_name = "tutorial_water_first"  # Name for this job

print("⚙️  Calculation Configuration")
print("="*50)
print(f"Method: {method}")
print(f"Basis Set: {basis}")
print(f"Job Name: {job_name}")
print("="*50)

print("\n📖 What is Hartree-Fock?")
print("   • Quantum method for calculating electron structure")
print("   • Solves Schrödinger equation approximately")
print("   • Gives total energy and molecular orbitals")
print("   • 'Self-Consistent Field' (SCF) iterative approach")

print("\n💡 What to expect:")
print("   • Energy: ~-76.03 Hartrees for H₂O/6-31G")
print("   • SCF iterations: typically 6-10")
print("   • Runtime: 5-10 minutes on cluster")

### ✅ Checkpoint 3

**Review your settings:**
- Method: **RHF** (good for H₂O)
- Basis: **6-31G** (medium quality)
- Job name: **tutorial_water_first** (you can change this)

**Key concept - What is a Hartree?**
- Unit of energy in quantum chemistry
- Named after Douglas Hartree
- 1 Hartree = 27.2 eV = 627.5 kcal/mol
- H₂O energy ~-76 Ha means 76 Hartrees below separated nuclei

---

## Step 5: Configure Resources

The calculation needs computational resources (CPUs, memory, time).

### Resource Settings
- **Cores:** Number of CPU processors (more = faster, but limited availability)
- **Memory:** RAM needed (more for larger molecules)
- **Walltime:** Maximum time allowed (job canceled if exceeded)

### For water molecule
Our small molecule doesn't need much:
- **2 cores** - Plenty for H₂O
- **1 GB memory** - More than enough
- **15 minutes** - Should finish in 5-10 min

### Run the cell below ⬇️

In [ ]:
# Resource configuration
cores = 2
memory_gb = 1
walltime = "00:15:00"  # HH:MM:SS format

print("💻 Computational Resources")
print("="*50)
print(f"CPU Cores: {cores}")
print(f"Memory: {memory_gb} GB")
print(f"Max Runtime: {walltime}")
print("="*50)

print("\n📊 Resource estimation:")
calculation = create_calculation(current_molecule, method, basis)
estimates = calculation.estimate_resources()

print(f"   Recommended cores: {estimates['cores']}")
print(f"   Recommended memory: {estimates['memory_gb']} GB")
print(f"   Estimated time: {estimates['walltime']}")

print("\n💡 Our settings match the recommendations!")

### ✅ Checkpoint 4

**Resource check:**
- Cores: **2** (sufficient)
- Memory: **1 GB** (sufficient)
- Time: **15 minutes** (generous buffer)

**Why these resources?**
- H₂O is tiny (only 10 electrons!)
- RHF/6-31G is fast method
- Could even use less, but safety buffer is good

---

## Step 6: Submit the Job! 🚀

Time to submit your first quantum chemistry calculation to the cluster!

### What happens when you submit?
1. QuantUI generates a Python script with your calculation
2. QuantUI generates a QuantUI submission script
3. Job is submitted to local execution
4. QuantUI finds available resources
5. Your calculation runs on a compute node
6. Results are saved to your directory

### Are you ready? Run the cell below! ⬇️

### ✅ Checkpoint 5

**Submission successful if you see:**
- ✅ "JOB SUBMITTED SUCCESSFULLY!"
- ✅ Internal ID (starts with your username)
- ✅ QuantUI Job ID (number)
- ✅ Status: "PENDING" or "RUNNING"

**Job States:**
- **PENDING** = Waiting in queue
- **RUNNING** = Currently calculating
- **COMPLETED** = Finished successfully
- **FAILED** = Something went wrong

---

## Step 7: Monitor Job Status

Let's check if your job is done!

### How to check status
- **Pending**: Waiting for resources
- **Running**: Calculation in progress
- **Completed**: Ready to view results!
- **Failed**: Error occurred (check error messages)

### Run the cell below (you can run it multiple times) ⬇️

### ✅ Checkpoint 6

**Wait until status is "COMPLETED"**

**While waiting (5-10 minutes):**
- ☕ Take a break
- 📖 Read ahead to Step 8
- 🤔 Think about: What energy do you expect?
- 🔄 Re-run this cell every minute or two

**Expected energy:** Around **-76.03 Hartrees**

---

## Step 8: View Results! 🎉

Your calculation is complete! Let's see what we learned about water.

### What results to look for:
1. **Total Energy** - How stable is the molecule?
2. **SCF Convergence** - Did the calculation succeed?
3. **Number of iterations** - How hard was it?
4. **Molecular orbitals** - Electron distributions

### Run the cell below ⬇️

In [ ]:
# View calculation results
print("📊 CALCULATION RESULTS")
print("="*70)

try:
    metadata = job_manager.get_job(tutorial_job_id)
    
    if metadata.status != "COMPLETED":
        print(f"⚠️  Job status: {metadata.status}")
        print("   Please wait for job to complete, then re-run this cell.")
    else:
        print(f"✓ Job: {metadata.job_name}")
        print(f"✓ Method: {metadata.method}/{metadata.basis}")
        print(f"✓ Molecule: {current_molecule.get_formula()}")
        print("="*70)
        
        # Get output files
        output_data = job_manager.get_job_output(metadata)
        
        if output_data['stdout']:
            output = output_data['stdout']
            
            # Extract key info
            print("\n🔬 KEY RESULTS:")
            print("-"*70)
            
            # Find total energy
            if "converged SCF energy" in output:
                for line in output.split('\n'):
                    if "converged SCF energy" in line:
                        energy_line = line
                        print(f"✓ {energy_line.strip()}")
                        
                        # Extract number
                        try:
                            energy = float(energy_line.split('=')[1].strip().split()[0])
                            print(f"\n💡 Interpreting the energy:")
                            print(f"   • Energy = {energy:.6f} Hartrees")
                            print(f"   • This is the total electronic energy")
                            print(f"   • More negative = more stable")
                            
                            # Compare to expected
                            expected = -76.03
                            diff = abs(energy - expected)
                            if diff < 0.1:
                                print(f"   ✅ Matches expected value (~{expected:.2f} Ha)")
                            else:
                                print(f"   ⚠️  Different from expected ({expected:.2f} Ha)")
                        except:
                            pass
                        break
            
            # Find iterations
            if "SCF converged in" in output:
                for line in output.split('\n'):
                    if "SCF converged in" in line:
                        print(f"\n✓ {line.strip()}")
                        print(f"   💡 Fewer iterations = easier convergence")
                        break
            
            print("\n" + "="*70)
            print("🎉 CALCULATION COMPLETED SUCCESSFULLY!")
            print("="*70)
            
        else:
            print("⚠️  No output available yet")
            
except Exception as e:
    print(f"❌ Error viewing results: {e}")
    import traceback
    traceback.print_exc()
# QuantUI — in-session calculation imports
from quantui import (
    Molecule, parse_xyz_input,
    MOLECULE_LIBRARY, SUPPORTED_METHODS, SUPPORTED_BASIS_SETS,
    session_can_handle, get_session_resources,
    PUBCHEM_AVAILABLE, VISUALIZATION_AVAILABLE, ASE_AVAILABLE,
)
try:
    from quantui import run_in_session, SessionResult
    PYSCF_AVAILABLE = True
except (ImportError, AttributeError):
    PYSCF_AVAILABLE = False
    print("PySCF not available — calculations will not run.")
    print("PySCF requires Linux, macOS, or WSL.")


### ✅ Final Checkpoint

**Success criteria:**
- ✅ Energy around **-76.03 Hartrees**
- ✅ SCF converged (not failed)
- ✅ Typically 6-10 iterations
- ✅ No error messages

**Understanding your results:**
- **Energy (-76.03 Ha)**: Total electronic energy of H₂O
- **SCF Convergence**: Iterative method found consistent solution
- **Hartrees**: Energy unit (1 Ha = 627 kcal/mol)

---

## 🎓 Learning Review

### What You Accomplished

1. ✅ Set up QuantUI environment
2. ✅ Defined a molecule (H₂O) with coordinates
3. ✅ Configured a Hartree-Fock calculation
4. ✅ Submitted job to QuantUI cluster
5. ✅ Monitored job status
6. ✅ Interpreted results

### Key Concepts Learned

**Molecule Definition:**
- XYZ coordinates specify 3D geometry
- Charge and multiplicity matter
- Validation prevents errors

**Hartree-Fock Method:**
- Quantum mechanical approach
- Self-Consistent Field (SCF) iterations
- Gives total energy and orbitals

**Basis Sets:**
- Mathematical functions for electrons
- Trade-off: accuracy vs. speed
- 6-31G is good medium choice

**QuantUI Job Management:**
- Submit to queue
- Monitor status
- Retrieve results

**Results Interpretation:**
- Energy in Hartrees
- More negative = more stable
- SCF convergence is success indicator

---

## 🚀 Next Steps

### What to try next:

1. **Modify and Resubmit**
   - Change job name
   - Re-run this tutorial with different molecule
   - Try NH₃ or CH₄

2. **Move to Tutorial 2**
   - Compare different basis sets
   - See accuracy vs. cost trade-offs
   - Learn about basis set convergence

3. **Experiment in Main Notebook**
   - Use [quantui_interface.ipynb](../quantui_interface.ipynb)
   - Try QuickStart Templates
   - Explore your own molecules

### Questions to Ponder

1. Why is water's energy negative?
2. What would happen with a bigger basis set?
3. How would methane (CH₄) be different?
4. What if we used the wrong multiplicity?

---

## 💬 Discussion Questions

Think about or discuss with classmates:

1. **Energy Meaning**: Why is the energy -76 Hartrees? What does the negative sign mean?

2. **Approximations**: Hartree-Fock is approximate. What physical effects does it miss? (Hint: electron correlation)

3. **Geometry**: How would results change if H-O-H angle was 180° (linear)?

4. **Basis Sets**: If 6-31G gives -76.03 Ha, what would cc-pVTZ give? Higher or lower?

5. **Applications**: How is this calculation useful? What can we learn about water?

---

## 📚 Further Reading

### Want to learn more?

**Hartree-Fock Method:**
- [Wikipedia: Hartree-Fock method](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method)
- Textbook: Szabo & Ostlund, "Modern Quantum Chemistry"

**Basis Sets:**
- [Basis Set Exchange](https://www.basissetexchange.org/)
- Review article: Jensen, "Atomic Orbital Basis Sets"

**PySCF Documentation:**
- [PySCF User Guide](https://pyscf.org/user.html)
- Example calculations and tutorials

**Quantum Chemistry Basics:**
- Levine, "Quantum Chemistry"
- Cramer, "Essentials of Computational Chemistry"

---

## ✅ Tutorial Complete!

**Congratulations on your first quantum chemistry calculation!** 🎉

You've taken the first step into computational chemistry. The skills you learned here will be the foundation for more complex calculations.

**Ready for more?** → [Tutorial 2: Basis Set Study](02_basis_set_study.ipynb)

---

*Have questions or found an issue? Contact your instructor or create an issue on GitHub.*